# Provision data quality report
**Author**:  Greg Slater <br>
**Date created**:  November 2024 <br>
**Dataset Scope**: ODP datasets <br>
**Report Type**: Ad-hoc <br>

**Purpose**: The purpose of this report is to measure the quality of the data that makes up each data provision on the platform, by applying a data quality framework that sets out criteria that must be met in order to reach one of 4 different quality levels. These levels are based around the quality requirements of the ODP software which uses platform data.

Note: the datasets included in this report are active resources of active endpoints. So where we have retired endpoints we may have data for a provision still appearing on the platform but it will not appear in this report. This means in the "Dataset quality scoring detail" table not all ODP provisions are present, 68 compared to 73. This is because there are 5 ODP providers where we don't have endpoints or HE conservation-area data, so there are no issues to display.

Future improvements:
* Error handling. Queries not working may break bits of the report. Not very high priority while report is more of a POC.
* Base tables. Expand summaries to full ODP provision, including where no data at all. This could be done by switching the `qual_cat_summary` table to be constructed from a base of the provision table, rather than `qual_all` (which only includes provisions with quality issues).
* Adding more quality checks. This depends on more checks going live in issues or expectations tables, but once they are should be easy to add extra criteria checks through the `qual_` table structure.
* Include data from old endpoints. This will need re-working of the base table query (from `fi.get_endpoint_res_issues()`) to include old endpoints and resources. Though this will add complexity to work out which are the "latest" endpoints and resources to include, especially for provisions with multiple endpoints. May be low priority.


### Data quality framework  
The table below visualises the framework that is used to assign a quality level to each ODP data provision. 

The criteria marked as "true" at each level must be met by a data provision in order for it to be scored at that level. The levels are cumulative, so all criteria must be met in order for a provision to be scored as *data that is trustworthy*. Where we have data from alternative providers (e.g. Historic England conservation-area data) the first criteria cannot be met so it is scored as the first quality level, *some data*.

![quality framework table](quality-framework-table.png)

In [2]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from datetime import datetime
import folium
import matplotlib

try:
  import mapclassify

except:
  print("installing mapclassify")
  !pip install mapclassify
  import mapclassify

td = datetime.today().strftime('%Y-%m-%d')

In [3]:
def save_util_file(file_name):

    if os.path.isfile(file_name) == False:
        url = f"https://raw.githubusercontent.com/digital-land/jupyter-analysis/refs/heads/main/reports/measure_odp_data_quality/{file_name}"
        !wget {url}
        print(f"downloaded {file_name} from github")

    else:
        print("file available locally")

for f in ["functions_core.py", "functions_import.py", "functions_transform.py"]:
    save_util_file(f)

import functions_core as fc
import functions_import as fi
import functions_transform as ft

file available locally
file available locally
file available locally


In [4]:
output_dir = "../../data/quality_report/"
os.makedirs(output_dir, exist_ok=True)

## 1. Import

In [ ]:
# Issue quality criteria lookup
lookup_issue_qual = fi.get_issue_quality_lookup()

# Provision lookups
lookup_provision_odp = fi.get_odp_provision_lookup()
lookup_provision_odp.rename(columns={"dataset" : "pipeline"}, inplace=True)

# Provision rule lookup - used to work out which datasets are "mandated" (statutory, or
# encouraged specifically for LPAs). This report now covers ODP datasets plus mandated ones.
provision_rule_df = fi.get_provision_rule_lookup()
mandated_pipelines = ft.get_mandated_pipelines(provision_rule_df)

# Dataset subset dict for chart
dataset_subset_dict = dict({
        "ODP" : ["conservation-area", "conservation-area-document", "article-4-direction-area", "article-4-direction", "listed-building-outline", "tree", "tree-preservation-zone", "tree-preservation-order"],
        "BFL" : ["brownfield-land"],
        "Developers" : ["developer-agreement", "developer-agreement-contribution", "developer-agreement-transaction"],
        "Mandated" : mandated_pipelines
    })

# Base table - queried directly from datasette over HTTP (paginated), not from a downloaded .db file,
# since direct .db downloads from datasette.planning.data.gov.uk are now blocked (403)
ep_res_issues = fi.get_endpoint_res_issues()


# Below is all extra for adding in the authoritative-source checks
# ---------------------------------------------------------------------------------------------------------
# Organisation lookups
lookup_org = fi.get_organisation_lookup()
lookup_org[["lpa_flag", "organisation_entity"]] = lookup_org[["lpa_flag", "organisation_entity"]].astype(int)

# LPA boundaries
lpa_gdf = fc.get_pdp_dataset("local-planning-authority", "geometry")

# Authoritative-source signal: each dataset's own entity table carries a platform-computed
# `quality` field per entity (authoritative/some/etc), keyed by organisation_entity. This is
# a much better signal than a geospatial join - it reflects the real source of the data actually
# held, not just who is registered as the expected provider. Queried per pipeline present in the
# data; pipelines without a usable entity/quality/organisation_entity column (e.g. pure
# reference/enum datasets) just come back empty and are treated as "not checked".
quality_priority_map = fi.get_quality_priority_lookup()

entity_quality_frames = []
for pipeline in ep_res_issues["pipeline"].unique():
    pipeline_entity_quality = fi.get_entity_quality_lookup(pipeline)
    if not pipeline_entity_quality.empty:
        entity_quality_frames.append(pipeline_entity_quality)

entity_quality_raw = pd.concat(entity_quality_frames, ignore_index=True)

## 2. Transform

In [6]:
# sort out LPA table for joining

# rename for easier joining
lpa_gdf.rename(
    columns = {
        'name':'lpa_name',
        'reference':'LPACD'
    }, 
        inplace=True)

# restrict LPAs to un-ended ones and join on organisation field
lpa_live_gdf = lpa_gdf[["LPACD", "geometry"]].merge(
    lookup_org[lookup_org["end_date"].isnull()][["LPACD", "organisation", "organisation_name", "organisation_entity"]],
    how = "inner",
    on = "LPACD"
)

# set up base table - will now include LPAs with no data, and outer join keeps in non-LPA provided dataset
base = lpa_live_gdf[["LPACD", "organisation"]].merge(
    ep_res_issues,
    how = "outer",
    on = "organisation"
)


In [7]:
# AUTHORITATIVE LOOKUP - flags whether a provision's data actually comes from the authoritative
# source, using each dataset's own entity-level `quality` field. This is a separate axis from the
# severity-based tables below, not folded into qual_all - see make_score_summary_table.

auth_lookup = ft.make_authoritative_lookup(entity_quality_raw, quality_priority_map, lookup_org)

# CA MATCH CHECK TABLE - flagging when conservation-area counts per LPA don't match manual count

qual_match = ft.make_ca_count_match_issues_table(lpa_live_gdf)

# LPA BOUNDARY CHECK TABLE - flagging when entities for a provision are beyond the expected LPA boundary

qual_bounds = ft.make_lpa_boundary_issues_table(lpa_live_gdf)

# ISSUES TABLE - flagging when provisions have data quality issues

qual_issues = ft.make_issues_input_table(base, lookup_issue_qual)


# # FRESHNESS TABLE - flagging when provisions haven't been updated in last year - not included in quality framework for now

# # create table of old resources and flag quality level as 5
# ep_res_fresh_qual = ft.make_freshness_input_table(ep_res_issues, age_days = 365)


# ALL QUALITY CATEGORIES TABLE - joining all records of severity-based quality categories into one long table
# (authoritative status is handled separately via auth_lookup, not concatenated in here)
qual_all = pd.concat([qual_match, qual_bounds, qual_issues])
qual_all.head()

,LPACD,organisation,organisation_name,collection,pipeline,quality_criteria,quality_level,issue_type
0,E60000004,local-authority:MDB,Middlesbrough Borough Council,NaN,brownfield-land,3 - entities within LPA boundary,3.0,NaN
1,E60000005,local-authority:NBL,Northumberland County Council,NaN,brownfield-land,3 - entities within LPA boundary,3.0,NaN
2,E60000005,local-authority:NBL,Northumberland County Council,NaN,listed-building-outline,3 - entities within LPA boundary,3.0,NaN
3,E60000005,local-authority:NBL,Northumberland County Council,NaN,tree-preservation-zone,3 - entities within LPA boundary,3.0,NaN
4,E60000005,local-authority:NBL,Northumberland County Council,NaN,tree,3 - entities within LPA boundary,3.0,NaN


In [8]:
# # store functions & arguments that return quality calculation data as a list of tuples 
# qual_calc_functions = [
#     (ft.make_freshness_input_table, [ep_res_issues, 365]),
#     (ft.make_issues_input_table, [ep_res_issues, lookup_issue_qual])
# ]

# tables = [func(*args) for func, args in qual_calc_functions if isinstance(func(*args), pd.DataFrame)]
# print(len(tables))

In [ ]:
# 0-6 scale: authoritative axis (confirmed LPA-sourced data?) crossed with rung axis
# (some data -> usable -> trustworthy). 0 is for provisions with no data at all - either no
# active endpoint, or an active endpoint that produced zero actual entities (see
# apply_zero_entity_override below).
level_map = {
    6: "6. authoritative trustworthy data",
    5: "5. authoritative usable data",
    4: "4. authoritative data",
    3: "3. non-authoritative trustworthy data",
    2: "2. non-authoritative usable data",
    1: "1. non-authoritative/some data",
    0: "0. no data"}


qual_summary = ft.make_score_summary_table(qual_all, auth_lookup, level_map)

# an active endpoint that produced zero entities has nothing meaningful for the severity/
# authoritative axes to score - force it to "no data" rather than whatever rung/authoritative
# combination the (empty) issue/entity metadata would otherwise imply
qual_summary = ft.apply_zero_entity_override(qual_summary, entity_quality_raw, lookup_org, level_map)

print(len(qual_summary))

## 3. Summarise

### ODP LPA x Dataset quality table

In [10]:

# subset to ODP and pivot
odp_lpa_summary = qual_summary.merge(
    lookup_provision_odp[["organisation", "pipeline", "cohort"]],
    how = "inner",
    on = ["organisation", "pipeline"]
)

odp_lpa_summary_wide = odp_lpa_summary.pivot(
    columns = "pipeline",
    values = "quality_level_label",
    index = ["cohort", "organisation", "organisation_name"]
).reset_index(
).sort_values(
    ["cohort", "organisation_name"]
)

odp_lpa_summary_wide.replace(np.nan, "0. no data", inplace=True)

In [11]:
# flag whether LPAs are "ready for ODP" (must be in the authoritative branch for all geography datasets)
# count and min quality of geography datasets for each provider
ready_for_odp_calc = qual_summary[qual_summary["pipeline"].isin(
    ["article-4-direction-area", "conservation-area", "listed-building-outline", "tree", "tree-preservation-zone"]
    )].groupby(
    ["organisation"], as_index=False
).agg(
    area_dataset_count = ("pipeline", "count"),
    min_quality_level = ("quality_level", "min")
)

# add flag - count == 5 means all datasets must be provided
# min_quality_level >= 4 means every geography dataset must be in the authoritative branch (4-6)
ready_for_odp_calc["ready_for_ODP_adoption"] = np.where(
    (ready_for_odp_calc["area_dataset_count"] == 5) &
    (ready_for_odp_calc["min_quality_level"] >= 4),
    "yes", "no"
)

# add flag to summary wide table
odp_lpa_summary_wide = odp_lpa_summary_wide.merge(
    ready_for_odp_calc[["organisation", "ready_for_ODP_adoption"]],
    how = "left",
    on = "organisation"
)

In [12]:
level_background_colours = {
    "6. authoritative trustworthy data" : "background-color: #1a6837",
    "5. authoritative usable data" : "background-color: #74c476",
    "4. authoritative data" : "background-color: #c7e9c0",
    "3. non-authoritative trustworthy data" : "background-color: #fed976",
    "2. non-authoritative usable data" : "background-color: #fd8d3c",
    "1. non-authoritative/some data" : "background-color: #e31a1c",
    "0. no data" : "background-color: #eaeaea"
    }

ready_flag_colours = {
        "yes" : "color:green"
    }

def make_color_mask_odp_lpa(df):
    #DataFrame with same index and columns names as original filled empty strings
    df_color_map =  pd.DataFrame("", index=df.index, columns=df.columns)

    flag_slice = df.columns[2:-1]
    for s in flag_slice:
        df_color_map[s] = df[s].map(level_background_colours)

    df_color_map["ready_for_ODP_adoption"] = df["ready_for_ODP_adoption"].map(ready_flag_colours)

    return df_color_map

# make_color_mask_odp_lpa(odp_lpa_summary)
# odp_lpa_summary.style.apply(make_color_mask_odp_lpa, axis=None)

### Dataset x quality categories table

In [13]:
# count issues by the quality category 
qual_cat_count = qual_all.groupby(
        ["pipeline", "organisation", "organisation_name", "quality_criteria"],
        as_index=False
    ).agg(
        n_issues = ("quality_level", "count")
    )

In [14]:
# create a base table with each quality category for each provision - this is so it can be pivoted correctly with all categories included
prov = qual_all[["pipeline", "organisation", "organisation_name"]].drop_duplicates()
prov["key"] = 1

qual_cat = qual_all[qual_all["quality_criteria"].notnull()][["quality_criteria"]].drop_duplicates()
qual_cat["key"] = 1

qual_cat_summary = prov.merge(
    qual_cat,
    how = "left",
    on = "key"
)
print(len(qual_cat_summary))

# left join on the counts to the base table
qual_cat_summary = qual_cat_summary.merge(
    qual_cat_count,
    how = "left",
    on = ['pipeline', 'organisation', 'organisation_name', 'quality_criteria']
)

# create boolean flag for each category
qual_cat_summary["issue_flag"] = np.where(qual_cat_summary["n_issues"] > 0, False, True)
print(len(qual_cat_summary))
# qual_cat_summary.head()

8394
8394


In [ ]:
# pivot quality category summary table so that quality categories are columns, join on overall quality level per provision
qual_cat_summary_wide = qual_cat_summary.pivot(
        columns = "quality_criteria",
        values = "issue_flag",
        index = ["pipeline", "organisation", "organisation_name"]
    ).reset_index(
    ).merge(
        qual_summary[["pipeline", "organisation", "quality_level_label"]],
        how = "left",
        on = ["pipeline", "organisation"]
    )

# bring in the authoritative-source check (separate axis, not one of the severity quality_criteria above)
qual_cat_summary_wide = qual_cat_summary_wide.merge(
    auth_lookup[["organisation", "pipeline", "is_authoritative", "authoritative_check_available"]],
    how = "left",
    on = ["organisation", "pipeline"]
)

# move quality_level_label to be the last column, rather than sitting right after the criteria columns
qual_cat_summary_wide = qual_cat_summary_wide[
    [c for c in qual_cat_summary_wide.columns if c != "quality_level_label"] + ["quality_level_label"]
]

def get_dataset_qual_detail(dataset):
    # just subsets and styles main wide quality detail table

    qual_detail = qual_cat_summary_wide[qual_cat_summary_wide["pipeline"] == dataset].copy()

    return qual_detail.style.apply(make_color_mask_dataset_lpa, axis=None)


flag_colours = {
        True : "color:green",
        False : "color:red"
    }

# named explicitly (rather than sliced by position) so this still works regardless of where
# extra columns like is_authoritative get merged in
criteria_cols = [c for c in qual_cat_summary_wide.columns if c in qual_cat["quality_criteria"].unique()]

def make_color_mask_dataset_lpa(df):
    #DataFrame with same index and columns names as original filled empty strings
    df_color_map =  pd.DataFrame("", index=df.index, columns=df.columns)
    # turn label column into colours
    df_color_map["quality_level_label"] = df["quality_level_label"].map(level_background_colours)

    for s in criteria_cols:
        df_color_map[s] = df[s].map(flag_colours)

    if "is_authoritative" in df.columns:
        df_color_map["is_authoritative"] = df["is_authoritative"].map(flag_colours)

    return df_color_map


# make widget - now covers ODP + Mandated datasets rather than ODP alone
dataset_dropdown = widgets.Dropdown(
    options = dataset_subset_dict["ODP"] + dataset_subset_dict["Mandated"],
    value = "article-4-direction",
    description = "Select Dataset: ",
)


### ODP quality maps

In [ ]:
level_colours = {
    "6. authoritative trustworthy data" : "#1a6837",
    "5. authoritative usable data" : "#74c476",
    "4. authoritative data" : "#c7e9c0",
    "3. non-authoritative trustworthy data" : "#fed976",
    "2. non-authoritative usable data" : "#fd8d3c",
    "1. non-authoritative/some data" : "#e31a1c",
    "0. no data" : "#eaeaea"
}

def map_odp_quality_scores(dataset):

    map_score = lpa_live_gdf.merge(
        qual_summary[qual_summary["pipeline"] == dataset][["LPACD", "pipeline", "quality_level_label"]],
        how = "left",
        on = "LPACD"
    )

    map_score["quality_level_label"] = map_score["quality_level_label"].replace(np.nan, "0. no data")
    map_score["colour"] = map_score["quality_level_label"].map(level_colours)
    map_score["geometry"] = map_score["geometry"].simplify(0.001)

    m = map_score.explore(
        tiles = "CartoDB positron",  # use "CartoDB positron" tiles
        popup = ["organisation_name", "pipeline", "quality_level_label"],  # add in field names to show in popups
        tooltip = False,
        color = map_score["colour"]
    )

    return display(m)


# make widget - now covers ODP + Mandated datasets rather than ODP alone
odp_dataset_list = dataset_subset_dict["ODP"] + dataset_subset_dict["Mandated"]

odp_dataset_dropdown = widgets.Dropdown(
    options = odp_dataset_list,
    value = "conservation-area",
    description = "Select Dataset: ",
)
# map_odp_quality_scores("tree")


### Chart

In [ ]:
# VISUALISE

def make_quality_overview_chart(subset):
    """
    Uses the qual summary table to display a horizontal bar chart 
    """

    qual_summary_subset = qual_summary[qual_summary["pipeline"].isin(dataset_subset_dict[subset])]

    # count providers by dataset & quality level
    qual_chart = qual_summary_subset.groupby(["pipeline", "quality_level", "quality_level_label"], as_index=False).agg(
        n_providers = ("quality_level", "count")
    )

    qual_chart.sort_values(["pipeline", "quality_level_label"], inplace=True)
    qual_chart_wide = qual_chart.pivot(columns = "quality_level_label", values = "n_providers", index = "pipeline")

    # colour by label rather than position, so whichever subset of levels actually appears
    # (including "0. no data", which can now occur for the zero-entity override) is coloured
    # correctly regardless of column order/count
    bar_colours = [level_background_colours[c].split(": ")[-1] for c in qual_chart_wide.columns]

    qual_chart_wide.plot.barh(
        stacked = True, 
        color = bar_colours, 
        figsize = (9, 6))

    # Add labels and title
    plt.xlabel('Count of providers')
    plt.ylabel('Dataset')
    plt.title('Quality levels for ODP datasets')
    plt.legend(title='Quality level')

    return plt.show()


# now exposes every group in dataset_subset_dict (ODP, BFL, Developers, Mandated), not just ODP
subset_dropdown = widgets.Dropdown(
    options = list(dataset_subset_dict.keys()),
    # value = dataset_list[0],
    description = "Select Dataset subset: ",
)

# widgets.interact(make_quality_overview_chart, subset = subset_dropdown)

## 4. Present

### Data quality overview chart - by dataset groups

In [18]:
widgets.interact(make_quality_overview_chart, subset = subset_dropdown)

interactive(children=(Dropdown(description='Select Dataset subset: ', options=('ODP',), value='ODP'), Output()…

<function __main__.make_quality_overview_chart(subset)>

### Data quality overview map - for ODP datasets

In [19]:
widgets.interact(map_odp_quality_scores, dataset = odp_dataset_dropdown)

interactive(children=(Dropdown(description='Select Dataset: ', options=('conservation-area', 'conservation-are…

<function __main__.map_odp_quality_scores(dataset)>

### ODP LPA overview table by dataset & quality

In [20]:
odp_lpa_summary_wide.style.apply(make_color_mask_odp_lpa, axis=None)

,cohort,organisation,organisation_name,article-4-direction,article-4-direction-area,conservation-area,conservation-area-document,listed-building-outline,tree,tree-preservation-order,tree-preservation-zone,ready_for_ODP_adoption
0,BOPS-Alpha,local-authority:COV,Coventry City Council,5. authoritative usable data,6. authoritative trustworthy data,6. authoritative trustworthy data,6. authoritative trustworthy data,5. authoritative usable data,5. authoritative usable data,5. authoritative usable data,6. authoritative trustworthy data,yes
1,BOPS-Beta,local-authority:BUC,Buckinghamshire Council,0. no data,4. authoritative data,6. authoritative trustworthy data,0. no data,0. no data,4. authoritative data,0. no data,4. authoritative data,no
2,BOPS-Beta,local-authority:NBL,Northumberland County Council,6. authoritative trustworthy data,6. authoritative trustworthy data,6. authoritative trustworthy data,6. authoritative trustworthy data,5. authoritative usable data,5. authoritative usable data,5. authoritative usable data,5. authoritative usable data,yes
3,BOPS-Discovery,local-authority:HCK,London Borough of Hackney,0. no data,0. no data,5. authoritative usable data,0. no data,0. no data,0. no data,0. no data,0. no data,no
4,BOPS-Discovery,local-authority:SWK,London Borough of Southwark,5. authoritative usable data,0. no data,6. authoritative trustworthy data,0. no data,5. authoritative usable data,5. authoritative usable data,0. no data,5. authoritative usable data,no
5,RIPA-Alpha,local-authority:CMD,London Borough of Camden,6. authoritative trustworthy data,6. authoritative trustworthy data,6. authoritative trustworthy data,6. authoritative trustworthy data,5. authoritative usable data,0. no data,0. no data,0. no data,no
6,RIPA-Alpha,local-authority:LBH,London Borough of Lambeth,5. authoritative usable data,5. authoritative usable data,6. authoritative trustworthy data,0. no data,5. authoritative usable data,4. authoritative data,4. authoritative data,0. no data,no
7,RIPA-Alpha,local-authority:LEW,London Borough of Lewisham,6. authoritative trustworthy data,5. authoritative usable data,6. authoritative trustworthy data,6. authoritative trustworthy data,6. authoritative trustworthy data,0. no data,0. no data,0. no data,no
8,RIPA-BOPS,local-authority:COL,Colchester City Council,6. authoritative trustworthy data,0. no data,6. authoritative trustworthy data,0. no data,5. authoritative usable data,5. authoritative usable data,0. no data,5. authoritative usable data,no
9,RIPA-BOPS,local-authority:DNC,Doncaster Metropolitan Borough Council,0. no data,6. authoritative trustworthy data,6. authoritative trustworthy data,0. no data,6. authoritative trustworthy data,6. authoritative trustworthy data,0. no data,5. authoritative usable data,yes


### Dataset quality scoring detail table

In [21]:
widgets.interact(get_dataset_qual_detail, dataset = dataset_dropdown)

interactive(children=(Dropdown(description='Select Dataset: ', index=3, options=('conservation-area', 'conserv…

<function __main__.get_dataset_qual_detail(dataset)>

### Output
Save report files

In [22]:
fn = os.path.join(output_dir, f"quality_ODP-dataset-scores-by-LPA_{td}.xlsx")
odp_lpa_summary_wide.style.apply(make_color_mask_odp_lpa, axis=None).to_excel(fn, index = False)

In [ ]:
fn = os.path.join(output_dir, f"quality_ODP-dataset-quality-detail_{td}.xlsx")

odp_qual_summary_out = qual_cat_summary_wide[
        qual_cat_summary_wide["pipeline"].isin(dataset_subset_dict["ODP"] + dataset_subset_dict["Mandated"])
    ].style.apply(make_color_mask_dataset_lpa, axis=None)

odp_qual_summary_out.to_excel(fn, index = False)